# Calculate percentage of columns covered by 70% in ProteinGym1
### A good MSA (similar to that from the EVmutation paper):
1. $>70\%$ of the columns covered by at least 70% of the sequences in the final alignment (can change this to 50/50)
2. Neff/L in 1-100, with some room for flexibility (<1 is ok after optimization)
3. If the two objectives above are conflicting, priority is given to 2
4. Additionally, for the purpose of PG2: each .m2a file should have
- more than 100 sequences
- less than 2M sequences (usually should be the case when Neff/L < 100)
- separately list the design sequences
5. Remove alignments that align to $<70\%$ of the length of the target sequence --> can change this to $<50\%$

In [9]:
import sys
from glob import glob
import numpy as np
from os.path import basename, dirname, exists, isdir
import pandas as pd
import os
import shutil
from collections import defaultdict
from scipy.stats import spearmanr

There are 2 directories with MSA results:
1. From Tranception - 152 entries (93 are also in list 2) - these have bitscore 0.1, ..., 0.9 - `_alignment_statistics.csv` doesn't have Neff/L and min colcov varies - have evcouplings models
2. From Navami - 186 entries (all the uniprot id in PG final reference file) - these have bitscore 0.03, 0.04, 0.05, 0.1, 0.3, 0.5 - `_alignment_statistics.csv` all have Neff/L and min colcov=50 - spearman already calculated in a separate file (downloaded)

They share 93 DMS_IDs and 95 proteins (the organism name may differ or may have a suffix in Tranception). The Tranception list also contains 9 ClinVar entries

In [4]:
# tranception (may be outdated)
msa_dir1 = "/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/"
navami_msa_dir1_file = '/n/groups/marks/projects/viral_families/notebooks/navami/followupNeffAnalysis/PG_analysis/full_neff_results_length.tsv'
# Navami (seqcov 50 colcov 50)
msa_dir2 = "/n/groups/marks/projects/viral_families/notebooks/navami/followupNeffAnalysis/PG_analysis/evh/output/uniref100"

## Calculate MSA stats, if missing

### Get the MSA stats from Navami's output

In [39]:
def aggregate_neff_l_colcov(msa_dir=msa_dir2):
    stats_df = {'DMS_ID': [], 'bitscore': [], 'num_seqs': [], 'Neff/L': [], 'Colcov50': []}
    for sample_path in glob(msa_dir + "/*"):
        sample = basename(sample_path)
        output_dirs = [p for p in glob(sample_path + '/' + sample + "_b*") if os.path.isdir(p)]
        for output_dir in output_dirs:
            output_name = basename(output_dir)  
            bitscore = output_name.removeprefix(f"{sample}_b")
            stat_file = output_dir + "/align/" + output_name + '_alignment_statistics.csv'
            try:
                stat_df = pd.read_csv(stat_file)
                neff, length, num_seqs, perc_cov = stat_df.iloc[0]["N_eff"], stat_df.iloc[0]["seqlen"], stat_df.iloc[0]["num_seqs"], stat_df.iloc[0]["perc_cov"]
                stats_df['DMS_ID'].append(sample)
                stats_df['bitscore'].append("0"+bitscore)
                stats_df['num_seqs'].append(num_seqs)
                stats_df['Neff/L'].append(round(neff/length, 3))
                stats_df['Colcov50'].append(perc_cov)
            except FileNotFoundError:
                continue
    return pd.DataFrame(stats_df)

In [25]:
msa_dir2_stats_df = aggregate_neff_l_colcov()

In [26]:
msa_dir2_stats_df

,DMS_ID,bitscore,num_seqs,Neff/L,Colcov50
0,A0A140D2T1_ZIKV_seqcov50_colcov50_theta0.99,0.03,10867,0.239,0.990
1,A0A140D2T1_ZIKV_seqcov50_colcov50_theta0.99,0.04,10875,0.242,0.989
2,A0A140D2T1_ZIKV_seqcov50_colcov50_theta0.99,0.05,10876,0.242,0.989
3,A0A140D2T1_ZIKV_seqcov50_colcov50_theta0.99,0.1,10877,0.241,0.989
4,A0A140D2T1_ZIKV_seqcov50_colcov50_theta0.99,0.3,10881,0.243,0.989
...,...,...,...,...,...
749,YNZC_BACSU_seqcov50_colcov50_theta0.8,0.04,209467,1603.033,0.641
750,YNZC_BACSU_seqcov50_colcov50_theta0.8,0.05,200782,1560.598,0.667
751,YNZC_BACSU_seqcov50_colcov50_theta0.8,0.1,193428,1449.589,0.641
752,YNZC_BACSU_seqcov50_colcov50_theta0.8,0.3,85737,490.299,0.667


In [27]:
msa_dir2_stats_df.to_csv("/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1_MSA_stats_Navami.csv", index=False)

### Get MSA stats from Tranception
about 90 (DMS_ID at various bitscores) of them are extracted by Navami. We have Spearman for 2 models and Neff/L.

Calculate column coverage for all

In [40]:
from data_utils import MSA_processing
def calculate_colcov(msa_location, 
                     threshold_sequence_frac_gaps=0.5,
                     threshold_focus_cols_frac_gaps=0.3):
    
    msa = MSA_processing(
        MSA_location=msa_location,
        threshold_sequence_frac_gaps=0.5,
        threshold_focus_cols_frac_gaps=0.5)
    
    perc_cov = round(len(msa.focus_cols) / len(msa.focus_seq), 4)
    return perc_cov

In [4]:
calculate_colcov('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/AMFR_HUMAN/AMFR_HUMAN_b09/align/AMFR_HUMAN_b09.a2m')

0.9149

In [41]:
def aggregate_colcov(input_df,
                     output_tsv,
                     threshold_sequence_frac_gaps=0.5,
                     threshold_focus_cols_frac_gaps=0.3):
    
    with open(output_tsv, 'w') as f_out:
        for _, row in input_df.iterrows():
            dms_id = row['dms_id']
            msa_location = row['full_path']
            try:
                perc_cov = calculate_colcov(msa_location)
            except TypeError:
                print(msa_location, "is not a str")
                continue
            neff_l = round(row['neff']/row['length'], 3)
            
            f_out.write('\t'.join([dms_id, msa_location, str(perc_cov), str(neff_l)])+'\n')

In [3]:
msa_neff_df = pd.read_csv(navami_msa_dir1_file, sep='\t')
# aggregate_colcov(msa_neff_df, '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1_MSA_stats_Tranception.tsv')

Got some MSA locations that are not strings but look like numbers. Check which a2ms are left out

In [4]:
def find_a2m_files_missing_from_tsv(tsv_path, alignment_runs_dir, path_col='full_path', save_to=None):
    """
    Finds all a2m files under alignment_runs_dir matching the pattern
    [DMS_ID]/[DMS_ID]_b[01-09]/align/[DMS_ID]_b[01-09].a2m, and reports
    which ones are NOT listed in the tsv's path_col.
    """
    df = pd.read_csv(tsv_path, sep='\t')
    tsv_paths = set(df[path_col])

    pattern = os.path.join(alignment_runs_dir, '*', '*_b*', 'align', '*.a2m')
    disk_paths = glob(pattern)

    missing_from_tsv = sorted(set(disk_paths) - tsv_paths)

    print(f"{len(disk_paths)} a2m files found on disk matching the expected layout")
    print(f"{len(missing_from_tsv)} of those are NOT listed in the tsv")

    if save_to:
        with open(save_to, 'w') as f:
            f.write('\n'.join(missing_from_tsv) + '\n')
        print(f"Saved list to {save_to}")

    return missing_from_tsv

In [5]:
missing = find_a2m_files_missing_from_tsv(
    navami_msa_dir1_file,
    msa_dir1,
    save_to = 'a2m_files_missing_from_tsv.txt'
)

1313 a2m files found on disk matching the expected layout
377 of those are NOT listed in the tsv
Saved list to a2m_files_missing_from_tsv.txt


In [6]:
missing

['/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/A0A3G4RHW3_ATRBE/A0A3G4RHW3_ATRBE_b01/align/A0A3G4RHW3_ATRBE_b01.a2m',
 '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/A0A3G4RHW3_ATRBE/A0A3G4RHW3_ATRBE_b02/align/A0A3G4RHW3_ATRBE_b02.a2m',
 '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/A0A3G4RHW3_ATRBE/A0A3G4RHW3_ATRBE_b03/align/A0A3G4RHW3_ATRBE_b03.a2m',
 '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/ARGR_ECO55/ARGR_ECO55_b02/align/ARGR_ECO55_b02.a2m',
 '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/B3RDS9_9INFA/B3RDS9_9INFA_b01/align/B3RDS9_9INFA_b01.a2m',
 '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/B3RDS9_9INFA/B3RDS9_9INFA_b02/align/B3RDS9_9INFA_b02.a2m',
 '/n

In [42]:
dms_id_set = set()
with open("a2m_files_missing_from_tsv.txt", "r") as file:
    missing = file.read().splitlines()
for msa_path in missing:
    dms_id = msa_path.removeprefix(msa_dir1)
    dms_id = dms_id.split('/')[0]
    dms_id_set.add(dms_id)
dms_id_set

{'A0A3G4RHW3_ATRBE',
 'ARGR_ECO55',
 'B3RDS9_9INFA',
 'BBC1_YEAST',
 'CASP3_HUMAN',
 'CASP7_HUMAN',
 'CATR_SCHDU',
 'CBX4_HUMAN',
 'CCR5_HUMAN',
 'CXCR4_HUMAN',
 'DNJA1_HUMAN',
 'DOCK1_HUMAN',
 'DYR_ECOLI',
 'ENVZ_ECOLI',
 'EPHB2_HUMAN',
 'FECA_ECOLI',
 'HEM3_HUMAN',
 'HSP82_YEAST',
 'I4EPC4_9INFA',
 'ILF3B_XENLA',
 'KCNJ2_MOUSE_uniref100_2022_08',
 'LACI_ECOLI',
 'MET_HUMAN',
 'MYO3_YEAST',
 'NKX31_HUMAN',
 'NRAM_I96A1',
 'NUSA_SALTY',
 'NUSG_MYCTU',
 'OBSCN_MOUSE',
 'ODP2_GEOSE',
 'OPSD_HUMAN',
 'OTC_HUMAN',
 'OTU7A_HUMAN',
 'OXDA_RHOTO',
 'PAI1_HUMAN',
 'PHOT_CHLRE',
 'PIN1_MACFA',
 'PITX2_HUMAN',
 'PKN1_MOUSE',
 'PPARG_HUMAN',
 'Q2N0S5_9HIV1',
 'Q53Z42_HUMAN',
 'Q837P4_ENTFA',
 'Q837P5_ENTFA',
 'Q8EG35_SHEON_uniref100_2022_08',
 'RAD_ANTMA',
 'RASK_HUMAN',
 'RASK_HUMAN_Ursu_2020',
 'RBP1_MOUSE',
 'RCRO_LAMBD',
 'RD23A_MOUSE',
 'RL20_AQUAE',
 'RNC_ECOLI',
 'RPC1_BP434',
 'RPC1_LAMBD',
 'S22A1_HUMAN',
 'SAV1_MOUSE',
 'SBI_STAAR',
 'SERC_HUMAN',
 'SOX30_MOUSE',
 'SPA_STAAU',
 'SPTN1_C

In [43]:
len(dms_id_set)

70

Claude compared which DMS IDs existing in PG1 (also Navami's list) appear in this:

In [44]:
dms_id_set_overlap_with_PG1 = {
    'BBC1_YEAST',
    'CASP3_HUMAN',
    'CASP7_HUMAN',
    'CBX4_HUMAN',
    'CCR5_HUMAN',
    'DNJA1_HUMAN',
    'DYR_ECOLI',
    'ENVZ_ECOLI',
    'EPHB2_HUMAN',
    'FECA_ECOLI',
    'HEM3_HUMAN',
    'HSP82_YEAST',
    'MET_HUMAN',
    'MYO3_YEAST',
    'NKX31_HUMAN',
    'NUSG_MYCTU',
    'ODP2_GEOSE',
    'OPSD_HUMAN',
    'OTC_HUMAN',
    'OTU7A_HUMAN',
    'OXDA_RHOTO',
    'PAI1_HUMAN',
    'PHOT_CHLRE',
    'PITX2_HUMAN',
    'PPARG_HUMAN',
    'Q2N0S5_9HIV1',
    'Q53Z42_HUMAN',
    'Q837P4_ENTFA',
    'Q837P5_ENTFA',
    'RAD_ANTMA',
    'RASK_HUMAN',
    'RCRO_LAMBD',
    'RL20_AQUAE',
    'RNC_ECOLI',
    'RPC1_BP434',
    'RPC1_LAMBD',
    'S22A1_HUMAN',
    'SAV1_MOUSE',
    'SERC_HUMAN',
    'SPA_STAAU',
    'SPTN1_CHICK',
    'SRBS1_HUMAN',
    'UBE4B_MOUSE',
    'VILI_CHICK',
}

In [45]:
def find_missing_bitscores(tsv_path, alignment_runs_dir, dms_ids, path_col='full_path'):
    """
    For each DMS_ID in dms_ids, finds which bitscore variants have an a2m
    file present on disk but are NOT listed in the tsv.

    Returns a dict: {dms_id: [missing_bitscores]}
    """
    df = pd.read_csv(tsv_path, sep='\t')
    tsv_paths = set(df[path_col])
    
    missing_msa_paths = []

    missing = defaultdict(list)
    for dms_id in dms_ids:
        pattern = os.path.join(alignment_runs_dir, dms_id, f'{dms_id}_b*', 'align', f'{dms_id}_b*.a2m')
        for path in glob(pattern):
            if path not in tsv_paths:
                missing_msa_paths.append(path)
                fname = os.path.basename(path)
                bitscore = fname.removeprefix(f"{dms_id}_b").removesuffix(".a2m")
                missing[dms_id].append(bitscore)

    for dms_id in sorted(missing):
        print(f"{dms_id}: missing bitscores {sorted(missing[dms_id])}")

    n_files = sum(len(v) for v in missing.values())
    print(f"\n{len(missing)} / {len(dms_ids)} DMS_IDs have at least one missing bitscore "
          f"({n_files} a2m files total)")

    return missing_msa_paths

In [46]:
missing_msa_paths = find_missing_bitscores(
    navami_msa_dir1_file,
    msa_dir1 ,
    dms_id_set_overlap_with_PG1
)

BBC1_YEAST: missing bitscores ['01', '02', '03', '04', '05', '06', '07', '08']
CASP3_HUMAN: missing bitscores ['01']
CASP7_HUMAN: missing bitscores ['01']
CBX4_HUMAN: missing bitscores ['01']
CCR5_HUMAN: missing bitscores ['01', '02', '03', '04', '05', '06', '07']
DNJA1_HUMAN: missing bitscores ['01', '02', '03', '04', '05', '06', '07', '08', '09']
DYR_ECOLI: missing bitscores ['01', '02', '03', '04', '05', '06', '07']
ENVZ_ECOLI: missing bitscores ['01', '02', '03', '04', '05', '06', '07']
EPHB2_HUMAN: missing bitscores ['01', '02', '03', '04', '05', '06']
FECA_ECOLI: missing bitscores ['01', '02', '03', '04']
HEM3_HUMAN: missing bitscores ['01', '02', '03', '04', '05', '06', '07', '08', '09']
HSP82_YEAST: missing bitscores ['01', '02', '03', '04', '05', '06']
MET_HUMAN: missing bitscores ['05', '06', '07', '08', '09']
MYO3_YEAST: missing bitscores ['01', '02', '03', '04', '05', '06', '07', '08']
NKX31_HUMAN: missing bitscores ['02', '03', '04', '05', '06', '07', '08', '09']
NUSG_MYCT

In [30]:
def add_missing_colcov(missing_msas,
                       output_tsv,
                       threshold_sequence_frac_gaps=0.5,
                       threshold_focus_cols_frac_gaps=0.3):
    
    with open(output_tsv, 'w') as f_out:
        f_out.write('\t'.join(['DMS_ID', 'MSA_path', 'Colcov50']) + '\n')
        for msa_location in missing_msas:
            dms_id = msa_location.removeprefix(msa_dir1)
            dms_id = dms_id.split('/')[0]
            try:
                perc_cov = calculate_colcov(msa_location)
            except TypeError:
                print(msa_location, "is not a str")
                continue
#             neff_l = round(row['neff']/row['length'], 3)
            
            f_out.write('\t'.join([dms_id, msa_location, str(perc_cov)])+'\n')

In [47]:
missing_msa_paths

['/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/RPC1_LAMBD/RPC1_LAMBD_b01/align/RPC1_LAMBD_b01.a2m',
 '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/RPC1_LAMBD/RPC1_LAMBD_b02/align/RPC1_LAMBD_b02.a2m',
 '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/RPC1_LAMBD/RPC1_LAMBD_b03/align/RPC1_LAMBD_b03.a2m',
 '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/RPC1_LAMBD/RPC1_LAMBD_b04/align/RPC1_LAMBD_b04.a2m',
 '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/RPC1_LAMBD/RPC1_LAMBD_b07/align/RPC1_LAMBD_b07.a2m',
 '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs/ENVZ_ECOLI/ENVZ_ECOLI_b01/align/ENVZ_ECOLI_b01.a2m',
 '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/

In [50]:
with open('PG1_Tranception_overlap_MSA_locations_missing_stats.csv', 'w') as f_out:
    for msa_path in missing_msa_paths:
        f_out.write(msa_path + '\n')

In [48]:
add_missing_colcov(missing_msa_paths, '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1_MSA_stats_Tranception_additional.tsv')

NameError: name 'add_missing_colcov' is not defined

The above maxed out memory. Will submit an AWS job.

1. Generate the mapping.csv file (see examples in `/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/Beltran_Lehner_2025/mappings`)
2. Upload the .a2m files in `PG1_Tranception_overlap_MSA_locations.csv`

In [55]:
def make_aws_mapping_csv(msa_paths, ref_file, out_csv):
    ref_df = pd.read_csv(ref_file)
    with open(out_csv, 'w') as f_out:
        f_out.write(','.join(['protein_name','msa_location','theta']) + '\n')
        for msa_path in msa_paths:
            dms_id = msa_path.removeprefix(msa_dir1)
            dms_id = dms_id.split('/')[0]
            theta = str(ref_df.loc[ref_df['UniProt_ID'] == dms_id, "MSA_theta"].iloc[0])
            if not theta:
                print(dms_id)
            f_out.write(','.join([dms_id, basename(msa_path), theta]) + '\n')

In [56]:
with open("PG1_Tranception_overlap_MSA_locations.csv", "r") as file:
    missing_msa_paths = file.read().splitlines()
make_aws_mapping_csv(missing_msa_paths, '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/reference_files/pg2_reference_current.csv', '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1_msa_stats_additional_mapping.csv')

## Check if, for Potts and PSSM, performance scales with Neff/L
Does it vary by coarse categories?

### Navami's set of 186 UniProt IDs

Navami's set in `navami_msa_dir1_file` didn't distinguish between different assays for the same UniProt ID. So I'll compute them again.

1. Create a symbolic link for all couplings models in 186 UniProt ID

In [58]:
pg1_models_dir = '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1/models'

In [59]:
!bash sym_link_models.sh

Copied: 1311 files


2. Use Claude to create a trimmed reference file of PG1: `/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/input/DMS_substitutions_PG1_trimmed.csv`
3. Score mutations with the models (positions $<50\%$ covered are not included in couplings models; update `calculations.py` to set those to NaN)

In [61]:
!bash score_mutants.sh


CondaError: Run 'conda init' before 'conda activate'

Traceback (most recent call last):
  File "./score_mutants_PG1.py", line 2, in <module>
    import pandas as pd
ModuleNotFoundError: No module named 'pandas'
Traceback (most recent call last):
  File "./score_mutants_PG1.py", line 2, in <module>
    import pandas as pd
ModuleNotFoundError: No module named 'pandas'


Some process got killed - need to run again

In [79]:
def check_missing_output(model_folder, output_score_folder, ref_df, model_suffix):
    
    for DMS_index in range(len(ref_df)):
        list_DMS = ref_df["DMS_id"]
        DMS_id = list_DMS[DMS_index]

        UniProt_ID = ref_df["UniProt_ID"][ref_df["DMS_id"] == DMS_id].values[0]
        model_pattern = f"{model_folder}/{UniProt_ID}{model_suffix}.model"
        matched_models = glob(model_pattern)

        if len(matched_models) > 0:
            for f in matched_models:
                bitscore = os.path.basename(f).split('_')[-1].removesuffix('.model')
                output_file = output_score_folder + DMS_id + "_" + bitscore + ".csv"
                if not os.path.isfile(output_file):
                    print(f"{output_file} is missing, DMS_index: {DMS_index}")
#         else:
#             print(f"No model file for: {DMS_id} (UniProt_ID: {UniProt_ID}, expected pattern: {model_pattern})")

In [86]:
check_missing_output(
    pg1_models_dir, 
    "/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1/all_EVmutation/", 
    pd.read_csv("/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/input/DMS_substitutions_PG1_trimmed.csv"),
    "_seqcov50_colcov50_theta*_b*"
)

4. Compute the Spearman for EVmutation and site independent

Check which output file has NaN for all model predictions

In [6]:
for f in glob(f"/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1/all_EVmutation/*.csv"):
    df = pd.read_csv(f)
    if df['prediction_independent'].isna().all():
        print(os.path.basename(f))

KCNH2_HUMAN_Kozek_2020_b.1.csv
KCNH2_HUMAN_Kozek_2020_b.3.csv
KCNH2_HUMAN_Kozek_2020_b.5.csv
LYAM1_HUMAN_Elazar_2016_b.3.csv
LYAM1_HUMAN_Elazar_2016_b.5.csv
RAF1_HUMAN_Zinkus-Boltz_2019_b.5.csv
SPG1_STRSG_Wu_2016_b.3.csv


In [10]:
def compute_spearman(df, score_col, dms_col="DMS_score"):
    valid = df[[score_col, dms_col]].dropna()
    rho, _ = spearmanr(valid[score_col], valid[dms_col])
    return rho

In [9]:
site_independent_spearman = []
evmutation_spearman = []
id_list = []
for f in glob(f"/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1/all_EVmutation/*.csv"):
    df = pd.read_csv(f)
    id_list.append(os.path.basename(f))
    site_independent_spearman.append(compute_spearman(df, "prediction_independent"))
    evmutation_spearman.append(compute_spearman(df, "prediction_epistatic"))

performance_df = pd.DataFrame({'id': id_list,
                               'EVmutation_spearman': evmutation_spearman,
                               'Site_independent_spearman': site_independent_spearman})

In [10]:
root = '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2'
performance_df.to_csv(root + '/EVCouplings/output/PG1/navami_all_evcouplings_spearman.csv', index=False)

### Tranception overlapping with PG1

1. Create a symbolic link for all couplings models

In [69]:
pg1_uniprot_id_set = set(pd.read_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/input/DMS_substitutions_PG1_trimmed.csv')['UniProt_ID'])
pg1_tranception_models = '/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1/Tranception_PG1_models'
os.makedirs(pg1_tranception_models, exist_ok=True)

In [70]:
for uniprot_id in pg1_uniprot_id_set:
    if os.path.isdir(msa_dir1 + uniprot_id):
        for model in glob(f"{msa_dir1}{uniprot_id}/{uniprot_id}_b*/couplings/{uniprot_id}_b*.model"):
            os.symlink(model, pg1_tranception_models + os.sep + basename(model))

2. Score mutations using the same input reference file above `/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/input/DMS_substitutions_PG1_trimmed.csv`
3. Check which jobs didn't finish

In [85]:
check_missing_output(
    pg1_tranception_models, 
    "/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1/Tranception_PG1_models_all_EVmutation/", 
    pd.read_csv("/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/input/DMS_substitutions_PG1_trimmed.csv"),
    "_b*"
)

In [7]:
for f in glob(f"/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1/Tranception_PG1_models_all_EVmutation/*.csv"):
    df = pd.read_csv(f)
    if df['prediction_independent'].isna().all():
        print(os.path.basename(f))

LYAM1_HUMAN_Elazar_2016_b02.csv
LYAM1_HUMAN_Elazar_2016_b03.csv
LYAM1_HUMAN_Elazar_2016_b04.csv
LYAM1_HUMAN_Elazar_2016_b05.csv
LYAM1_HUMAN_Elazar_2016_b06.csv
LYAM1_HUMAN_Elazar_2016_b07.csv
LYAM1_HUMAN_Elazar_2016_b08.csv
LYAM1_HUMAN_Elazar_2016_b09.csv
SPIKE_SARS2_Starr_2020_binding_b02.csv
SPIKE_SARS2_Starr_2020_expression_b02.csv
UBE4B_MOUSE_Starita_2013_b01.csv
UBE4B_MOUSE_Starita_2013_b02.csv
UBE4B_MOUSE_Starita_2013_b03.csv
UBE4B_MOUSE_Starita_2013_b04.csv
UBE4B_MOUSE_Starita_2013_b05.csv
UBE4B_MOUSE_Starita_2013_b06.csv
UBE4B_MOUSE_Starita_2013_b07.csv
UBE4B_MOUSE_Starita_2013_b08.csv
UBE4B_MOUSE_Starita_2013_b09.csv


In [11]:
site_independent_spearman = []
evmutation_spearman = []
id_list = []
for f in glob(f"/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1/Tranception_PG1_models_all_EVmutation/*.csv"):
    df = pd.read_csv(f)
    id_list.append(os.path.basename(f))
    site_independent_spearman.append(compute_spearman(df, "prediction_independent"))
    evmutation_spearman.append(compute_spearman(df, "prediction_epistatic"))

performance_df = pd.DataFrame({'id': id_list,
                               'EVmutation_spearman': evmutation_spearman,
                               'Site_independent_spearman': site_independent_spearman})

In [13]:
performance_df.to_csv('/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1/tranception_all_evcouplings_spearman.csv', index=False)

Reran all the scoring jobs at higher memory without any output still

### Looking at Domainome

Check the last section of `PG2/EVCouplings/scripts/Beltran_Lehner_2025/EVmutation_scoring.ipynb`

## Check if we need to update selected bitscores